In [14]:
import numpy as np
import time
import sys
import math

def load_tsp(filename):
    coords = []
    edge_weight_type = "EUC_2D"
    edge_weight_format = None
    edge_values = []

    mode = "HEADER"

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if ":" in line and mode == "HEADER":
                key, value = [x.strip() for x in line.split(":", 1)]

                if key == "EDGE_WEIGHT_TYPE":
                    edge_weight_type = value.upper()

                elif key == "EDGE_WEIGHT_FORMAT":
                    edge_weight_format = value.upper()

                continue

            if line == "NODE_COORD_SECTION" or line == "DISPLAY_DATA_SECTION":
                mode = "COORD"
                continue

            if line == "EDGE_WEIGHT_SECTION":
                mode = "EDGE"
                continue

            if line == "EOF":
                break

            if mode == "COORD":
                parts = line.split()
                nums = []

                for x in parts:
                    try:
                        nums.append(float(x))
                    except:
                        pass

                if len(nums) == 3:
                    coords.append(nums[1:])
                elif len(nums) == 2:
                    coords.append(nums)

            elif mode == "EDGE":
                edge_values.extend(map(int, line.split()))

    return (
        np.array(coords),
        edge_weight_type,
        edge_weight_format,
        edge_values,
    )


def distance_matrix(cities, edge_weight_type,
                    edge_weight_format=None,
                    edge_values=None):
    n = len(cities)
    D = np.zeros((n, n), dtype=float)

    # ---EXPLICIT matrices---
    if edge_weight_type == "EXPLICIT":
        # Compute n from edge_values
        if edge_weight_format == "LOWER_DIAG_ROW":
            n = int((math.sqrt(8 * len(edge_values) + 1) - 1) / 2)
        elif edge_weight_format in ("UPPER_ROW", "LOWER_ROW"):
            n = int((1 + math.sqrt(1 + 8 * len(edge_values))) / 2)
        elif edge_weight_format == "FULL_MATRIX":
            n = int(math.sqrt(len(edge_values)))
        else:
            raise NotImplementedError(f"EXPLICIT format {edge_weight_format} not implemented")

        D = np.zeros((n, n), dtype=int)
        k = 0

        if edge_weight_format == "LOWER_DIAG_ROW":
            for i in range(n):
                for j in range(i + 1):
                    D[i][j] = edge_values[k]
                    D[j][i] = edge_values[k]
                    k += 1
        elif edge_weight_format == "UPPER_ROW":
            for i in range(n):
                for j in range(i + 1, n):
                    D[i][j] = edge_values[k]
                    D[j][i] = edge_values[k]
                    k += 1
        elif edge_weight_format == "LOWER_ROW":
            for i in range(n):
                for j in range(i):
                    D[i][j] = edge_values[k]
                    D[j][i] = edge_values[k]
                    k += 1
        elif edge_weight_format == "FULL_MATRIX":
            for i in range(n):
                for j in range(n):
                    D[i][j] = edge_values[k]
                    k += 1
        return D

    # ----Geometric distances---
    def round_dist(d):
        return int(d + 0.5)

    def geo_to_rad(x):
        deg = int(x)
        minute = x - deg
        return math.pi * (deg + 5.0 * minute / 3.0) / 180.0

    def geo_distance(coord1, coord2):
        lati = geo_to_rad(coord1[0])
        loni = geo_to_rad(coord1[1])
        latj = geo_to_rad(coord2[0])
        lonj = geo_to_rad(coord2[1])

        q1 = math.cos(loni - lonj)
        q2 = math.cos(lati - latj)
        q3 = math.cos(lati + latj)

        return int(
            6378.388 *
            math.acos(
                0.5 * ((1 + q1) * q2 - (1 - q1) * q3)
            ) + 1.0
        )

    def att_distance(coord1, coord2):
        xd = coord1[0] - coord2[0]
        yd = coord1[1] - coord2[1]
        rij = math.sqrt((xd * xd + yd * yd) / 10.0)
        tij = int(rij)
        if tij < rij:
            dij = tij + 1
        else:
            dij = tij
        return dij

    for i in range(n):
        for j in range(i+1, n):
            if edge_weight_type == "EUC_2D":
                d = round_dist(np.linalg.norm(cities[i] - cities[j]))
            elif edge_weight_type == "EUC_3D":
                d = round_dist(np.linalg.norm(cities[i] - cities[j]))
            elif edge_weight_type == "MAN_2D":
                d = round_dist(np.sum(np.abs(cities[i] - cities[j])))
            elif edge_weight_type == "MAN_3D":
                d = round_dist(np.sum(np.abs(cities[i] - cities[j])))
            elif edge_weight_type == "MAX_2D":
                d = round_dist(np.max(np.abs(cities[i] - cities[j])))
            elif edge_weight_type == "MAX_3D":
                d = round_dist(np.max(np.abs(cities[i] - cities[j])))
            elif edge_weight_type == "CEIL_2D":
                d = math.ceil(np.linalg.norm(cities[i] - cities[j]))
            elif edge_weight_type == "ATT":
                d = att_distance(cities[i], cities[j])
            elif edge_weight_type == "GEO":
                d = geo_distance(cities[i], cities[j])
            else:
                raise NotImplementedError(f"Edge weight type {edge_weight_type} not implemented")
            D[i][j] = D[j][i] = d

    # Cast to int if all distances are integral
    if np.all(D == D.astype(int)):
        D = D.astype(int)
    return D


def classical_solver(D):
    tour = nearest_neighbor(D)
    history = [tour_length(tour, D)]
    while True:
        new_tour, new_len = two_opt_pass(tour, D)
        if new_len < history[-1]:
            tour = new_tour
            history.append(new_len)
        else:
            break
    return tour, history[-1], history

In [15]:
import numpy as np

class QSC:
    # Phase 1: Quantum Swarm Core (cos² probability, adaptive learning, noise)
    def __init__(self, D, pop_size=30, iterations=100,
                 base_step=0.05, decay=0.98, noise_std=0.01, seed=None):
        self.D = D
        self.n = len(D)
        self.pop_size = pop_size
        self.iterations = iterations
        self.base_step = base_step
        self.decay = decay
        self.noise_std = noise_std
        self.rng = np.random.default_rng(seed)

        self.theta = np.full((pop_size, self.n), np.pi/4)
        self.best_theta = None
        self.best_tour = None
        self.best_cost = float('inf')
        self.history = []

    def tour_length(self, tour):
        # vectorised tour length
        tour = np.asarray(tour)
        return self.D[tour[:-1], tour[1:]].sum() + self.D[tour[-1], tour[0]]

    def decode(self, theta):
        probs = np.cos(theta) ** 2
        probs += self.rng.normal(0, 1e-6, self.n)
        ranking = np.argsort(-probs)
        return ranking.tolist()

    def step_size(self, iteration):
        return self.base_step * (self.decay ** iteration)

    def run(self, verbose=False):
        for i in range(self.pop_size):
            tour = self.decode(self.theta[i])
            cost = self.tour_length(tour)
            if cost < self.best_cost:
                self.best_cost = cost
                self.best_tour = tour.copy()
                self.best_theta = self.theta[i].copy()
        self.history.append(self.best_cost)

        for it in range(1, self.iterations):
            step = self.step_size(it)
            for i in range(self.pop_size):
                direction = np.sign(self.best_theta - self.theta[i])
                self.theta[i] += step * direction
                self.theta[i] += self.rng.normal(0, self.noise_std * step, self.n)
                self.theta[i] = np.clip(self.theta[i], 0, np.pi/2)

                tour = self.decode(self.theta[i])
                cost = self.tour_length(tour)
                if cost < self.best_cost:
                    self.best_cost = cost
                    self.best_tour = tour.copy()
                    self.best_theta = self.theta[i].copy()
            self.history.append(self.best_cost)

            if verbose and it % 10 == 0:
                print(f"QSC iter {it:3d}/{self.iterations}  best={self.best_cost:.4f}")

        return self.best_tour, self.best_cost, self.best_theta, self.history


import numpy as np

class QEC:
    # Phase 2: Quantum Evolutionary Core (rotation gate, mutation, elitism)
    def __init__(self, D, seed_theta, seed_tour,
                 pop_size=40, generations=150,
                 rotation_step=0.05, mutation_rate=0.02, seed=None):
        self.D = D
        self.n = len(D)
        self.pop_size = pop_size
        self.generations = generations
        self.base_step = rotation_step
        self.mutation_rate = mutation_rate
        self.rng = np.random.default_rng(seed)

        # initialise population around seed
        self.theta = np.tile(seed_theta, (pop_size, 1))
        self.theta += self.rng.normal(0, 0.02, self.theta.shape)
        self.theta = np.clip(self.theta, 0, np.pi/2)

        self.pbest_theta = self.theta.copy()
        self.pbest_tours = [None] * pop_size
        self.pbest_costs = np.full(pop_size, np.inf)

        self.best_theta = seed_theta.copy()
        self.best_tour = seed_tour.copy()
        self.best_cost = self.tour_length(seed_tour)
        self.history = []

    # ------ TSP helpers---------
    def tour_length(self, tour):
        tour = np.asarray(tour)
        return self.D[tour[:-1], tour[1:]].sum() + self.D[tour[-1], tour[0]]

    def decode(self, theta):
        probs = np.cos(theta) ** 2
        probs += self.rng.normal(0, 1e-12, self.n)
        ranking = np.argsort(-probs)
        return ranking.tolist()

    # --- Adaptive rotation magnitude ---
    def _adaptive_step(self, generation):
        decay = 1.0 - generation / self.generations
        return self.base_step * (0.2 + 0.8 * decay) * np.pi

    # --- Quantum rotation gate ---
    def rotation_gate(self, i, current_tour, current_cost, generation):
        step = self._adaptive_step(generation)

        p_tour = self.pbest_tours[i]
        p_cost = self.pbest_costs[i]
        g_tour = self.best_tour
        g_cost = self.best_cost

        if p_cost < current_cost:
            for pos in range(self.n):
                c_curr = current_tour[pos]
                c_p = p_tour[pos]
                if c_curr != c_p:
                    self.theta[i, c_p] -= step
                    self.theta[i, c_curr] += step

        if g_cost < current_cost:
            for pos in range(self.n):
                c_curr = current_tour[pos]
                c_g = g_tour[pos]
                if c_curr != c_g:
                    self.theta[i, c_g] -= step
                    self.theta[i, c_curr] += step

        self.theta[i] = np.clip(self.theta[i], 0, np.pi/2)

    # --- Quantum mutation ---
    def mutation(self, i):
        mask = self.rng.random(self.n) < self.mutation_rate
        self.theta[i, mask] += self.rng.normal(0, 0.02 * np.pi, size=mask.sum())
        self.theta[i] = np.clip(self.theta[i], 0, np.pi/2)

    # --- Evaluation & update ---
    def evaluate_and_update(self, i):
        tour = self.decode(self.theta[i])
        cost = self.tour_length(tour)

        if cost < self.pbest_costs[i]:
            self.pbest_costs[i] = cost
            self.pbest_tours[i] = tour.copy()
            self.pbest_theta[i] = self.theta[i].copy()

        if cost < self.best_cost:
            self.best_cost = cost
            self.best_tour = tour.copy()
            self.best_theta = self.theta[i].copy()

        return tour, cost

    # --- Elitism ---
    def apply_elitism(self):
        worst_idx = np.argmax(self.pbest_costs)
        self.theta[worst_idx] = self.best_theta.copy()
        self.pbest_theta[worst_idx] = self.best_theta.copy()
        self.pbest_tours[worst_idx] = self.best_tour.copy()
        self.pbest_costs[worst_idx] = self.best_cost

    # --- Population statistics ---
    def evaluate_population(self):
        tours = []
        costs = np.zeros(self.pop_size)
        for i in range(self.pop_size):
            tours.append(self.decode(self.theta[i]))
            costs[i] = self.tour_length(tours[-1])
        return tours, costs

    def population_statistics(self):
        _, costs = self.evaluate_population()
        return {
            "best": float(np.min(costs)),
            "mean": float(np.mean(costs)),
            "worst": float(np.max(costs)),
            "std": float(np.std(costs))
        }

    # --- Generation cycle ----
    def evolve_one_generation(self, generation):
        for i in range(self.pop_size):
            current_tour = self.decode(self.theta[i])
            current_cost = self.tour_length(current_tour)
            self.rotation_gate(i, current_tour, current_cost, generation)
            self.mutation(i)
            self.evaluate_and_update(i)

        self.apply_elitism()
        self.history.append(self.best_cost)

    def run(self, verbose=False):
        for i in range(self.pop_size):
            self.evaluate_and_update(i)
        self.history.append(self.best_cost)

        for generation in range(self.generations):
            self.evolve_one_generation(generation)
            if verbose:
                pass

        tours = [self.decode(theta) for theta in self.theta]
        return (
            self.best_tour.copy(),
            self.best_cost,
            self.best_theta.copy(),
            self.theta.copy(),
            tours,
            self.history.copy()
        )

    # --- Accessors ---
    def get_best_tour(self):
        return self.best_tour.copy()
    def get_best_cost(self):
        return self.best_cost
    def get_best_theta(self):
        return self.best_theta.copy()
    def get_population(self):
        return self.theta.copy()
    def get_history(self):
        return self.history.copy()

    # --- Validation ---
    def validate_population(self):
        if np.any(self.theta < 0):
            raise ValueError("Negative quantum angle detected.")
        if np.any(self.theta > np.pi/2):
            raise ValueError("Quantum angle exceeds π/2.")
        return True

    def validate_best(self):
        if self.best_tour is None:
            raise ValueError("Best tour is not initialized.")
        if len(self.best_tour) != self.n:
            raise ValueError("Invalid tour length.")
        if sorted(self.best_tour) != list(range(self.n)):
            raise ValueError("Best tour is not a valid permutation.")
        return True

    def summary(self):
        pass

    def diagnostics(self):
        pass


import numpy as np
import random
import math

class DAC:
    # Phase 3: Differential Evolution & Annealing Core (DE + SA + 2-opt)
    def __init__(self, D, population, generations=100,
                 temperature=100.0, cooling=0.995,
                 two_opt_interval=10, seed=None):
        self.D = D
        self.n = len(D)
        self.generations = generations
        self.temperature = temperature
        self.cooling = cooling
        self.two_opt_interval = two_opt_interval

        self.rng = np.random.default_rng(seed)
        self.random_gen = random.Random(seed)

        self.population = [tour.copy() for tour in population]

        costs = [self.tour_length(t) for t in self.population]
        best_index = np.argmin(costs)
        self.best_tour = self.population[best_index].copy()
        self.best_cost = costs[best_index]
        self.history = []

    # --- TSP helpers ---
    def tour_length(self, tour):
        tour = np.asarray(tour)
        return self.D[tour[:-1], tour[1:]].sum() + self.D[tour[-1], tour[0]]

    def swap(self, tour, i, j):
        child = tour.copy()
        child[i], child[j] = child[j], child[i]
        return child

    def random_swap(self, tour):
        i, j = self.random_gen.sample(range(self.n), 2)
        return self.swap(tour, i, j)

    # --- 2-opt first improvement ---
    def two_opt_pass(self, tour):
        best = tour.copy()
        best_cost = self.tour_length(best)
        improved = False
        for i in range(1, self.n - 2):
            for j in range(i + 1, self.n):
                if j - i == 1:
                    continue
                candidate = best[:i] + best[i:j][::-1] + best[j:]
                cost = self.tour_length(candidate)
                if cost < best_cost:
                    best = candidate
                    best_cost = cost
                    improved = True
        return best, best_cost, improved

    # --- Parent selection --
    def select_parents(self, current_index):
        indices = list(range(len(self.population)))
        indices.remove(current_index)
        r1, r2, r3 = self.random_gen.sample(indices, 3)
        return self.population[r1], self.population[r2], self.population[r3]

    # --- Differential mutation (permutation‑preserving) ---
    def differential_mutation(self, target, parent1, parent2, parent3):
        mutant = target.copy()
        differing_positions = [i for i in range(self.n) if parent2[i] != parent3[i]]
        if not differing_positions:
            return mutant

        self.random_gen.shuffle(differing_positions)
        mutation_factor = max(1, len(differing_positions) // 2)
        selected = differing_positions[:mutation_factor]

        for position in selected:
            desired_city = parent1[position]
            current_position = mutant.index(desired_city)
            mutant[position], mutant[current_position] = mutant[current_position], mutant[position]
        return mutant

    # --- Order Crossover (OX) ---
    def crossover(self, target, mutant):
        start, end = sorted(self.random_gen.sample(range(self.n), 2))
        child = [-1] * self.n
        child[start:end] = mutant[start:end]

        pointer = end
        for city in target:
            if city in child:
                continue
            while child[pointer % self.n] != -1:
                pointer += 1
            child[pointer % self.n] = city
        return child

    # -- Simulated Annealing acceptance ----
    def accept(self, current_cost, candidate_cost):
        if candidate_cost <= current_cost:
            return True
        if self.temperature <= 1e-12:
            return False
        delta = candidate_cost - current_cost
        probability = math.exp(-delta / self.temperature)
        return self.random_gen.random() < probability

    def cool(self):
        self.temperature *= self.cooling

    # -- Individual evolution ---
    def evolve_individual(self, index):
        target = self.population[index]
        current_cost = self.tour_length(target)
        p1, p2, p3 = self.select_parents(index)
        mutant = self.differential_mutation(target, p1, p2, p3)
        child = self.crossover(target, mutant)
        child_cost = self.tour_length(child)
        if self.accept(current_cost, child_cost):
            self.population[index] = child
            current_cost = child_cost
        if current_cost < self.best_cost:
            self.best_cost = current_cost
            self.best_tour = child.copy()

    # -- Full generation (2‑opt on global best only) --
    def evolve_generation(self, gen):
        for i in range(len(self.population)):
            self.evolve_individual(i)

        if (gen + 1) % self.two_opt_interval == 0:
            improved = True
            cur = self.best_tour.copy()
            cur_cost = self.best_cost
            passes = 0
            while improved and passes < 3:
                cur, cur_cost, improved = self.two_opt_pass(cur)
                passes += 1
            if cur_cost < self.best_cost:
                self.best_cost = cur_cost
                self.best_tour = cur.copy()

        self.cool()
        self.history.append(self.best_cost)

    def run(self, verbose=False):
        self.history.append(self.best_cost)
        for gen in range(self.generations):
            self.evolve_generation(gen)
            if verbose:
                pass
        return self.best_tour.copy(), self.best_cost, self.history.copy()

    # --- Accessors --
    def get_best_tour(self):
        return self.best_tour.copy()
    def get_best_cost(self):
        return self.best_cost
    def get_history(self):
        return self.history.copy()
    def get_population(self):
        return [tour.copy() for tour in self.population]

    # --- Validation --
    def validate_population(self):
        for idx, tour in enumerate(self.population):
            if len(tour) != self.n:
                raise ValueError(f"Tour {idx} has incorrect length.")
            if sorted(tour) != list(range(self.n)):
                raise ValueError(f"Tour {idx} is not a valid permutation.")
        return True

    def validate_best(self):
        if self.best_tour is None:
            raise ValueError("Best tour has not been initialized.")
        if len(self.best_tour) != self.n:
            raise ValueError("Best tour length is invalid.")
        if sorted(self.best_tour) != list(range(self.n)):
            raise ValueError("Best tour is not a valid permutation.")
        return True

    def diagnostics(self):
        pass

    def summary(self):
        pass


def qmh_solver_with_seed(D, seed=None):
    # Same as qmh_solver but with explicit seed
    qsc = QSC(D, seed=seed)
    qsc_tour, qsc_cost, qsc_theta, h1 = qsc.run()
    qec = QEC(D, qsc_theta, qsc_tour, seed=seed)
    qec_tour, qec_cost, qec_theta, qec_theta_population, qec_population, h2 = qec.run()
    dac = DAC(D, qec_population, seed=seed)
    best_tour, best_cost, h3 = dac.run()
    return best_tour, best_cost, h1, h2, h3

In [16]:
import time
import csv
import numpy as np

# TSP instances to test
filenames = [
    #"att48.tsp",
    "bayg29.tsp",
    #"burma14.tsp",
]

NUM_QMH_RUNS = 5

results = []

for fname in filenames:
    print(f"\nProcessing {fname} ...")
    coords, etype, eformat, evalues = load_tsp(fname)
    D = distance_matrix(coords, etype, eformat, evalues)

    # Classical solver
    start = time.time()
    c_tour, c_cost, c_history = classical_solver(D)
    c_time = time.time() - start

    # QMH solver (multiple runs)
    q_costs = []
    q_times = []
    q_best_cost = float('inf')
    q_best_time = None
    q_best_tour = None

    for run in range(NUM_QMH_RUNS):
        seed = 42 + run
        start = time.time()
        q_tour, q_cost, h1, h2, h3 = qmh_solver_with_seed(D, seed=seed)
        elapsed = time.time() - start
        q_costs.append(q_cost)
        q_times.append(elapsed)
        if q_cost < q_best_cost:
            q_best_cost = q_cost
            q_best_time = elapsed
            q_best_tour = q_tour

    # Statistics
    q_avg_cost = np.mean(q_costs)
    q_std_cost = np.std(q_costs)
    q_avg_time = np.mean(q_times)
    q_std_time = np.std(q_times)
    q_min_time = np.min(q_times)
    imp_best = ((c_cost - q_best_cost) / c_cost) * 100
    imp_avg  = ((c_cost - q_avg_cost) / c_cost) * 100

    results.append({
        'instance': fname.replace('.tsp', ''),
        'classical_cost': c_cost,
        'classical_time': c_time,
        'q_best_cost': q_best_cost,
        'q_avg_cost': q_avg_cost,
        'q_std_cost': q_std_cost,
        'q_best_time': q_best_time,
        'q_avg_time': q_avg_time,
        'q_std_time': q_std_time,
        'q_min_time': q_min_time,
        'improvement_best_%': imp_best,
        'improvement_avg_%': imp_avg
    })

    print(f"  Classical: {c_cost:.2f} in {c_time:.3f}s")
    print(f"  QMH best : {q_best_cost:.2f} in {q_best_time:.3f}s (avg {q_avg_cost:.2f} ± {q_std_cost:.2f})")
    print(f"  Improvement (best): {imp_best:.2f}%")

# Save results to CSV
csv_fields = [
    'instance', 'classical_cost', 'classical_time',
    'q_best_cost', 'q_avg_cost', 'q_std_cost',
    'q_best_time', 'q_avg_time', 'q_std_time', 'q_min_time',
    'improvement_best_%', 'improvement_avg_%'
]

with open('results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print("\nResults saved to results.csv")


Processing bayg29.tsp ...
  Classical: 1762.00 in 0.030s
  QMH best : 1628.00 in 1.060s (avg 1673.80 ± 41.05)
  Improvement (best): 7.60%

Results saved to results.csv
